# 02 — Baseline model

Builds and solves the MILP of [`docs/formulation.md`](../docs/formulation.md) with Gurobi for one synthetic instance. Each constraint has its own cell, preceded by its equation and operational meaning.

## 1. Setup

In [ ]:
from pathlib import Path

import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB

DATA_DIR = Path("..") / "data" / "synthetic"
SCENARIO = "A"  # "A": current regime (RD 1146/2006); "B": RD 951/2026 regime

## 2. Load instance

In [ ]:
# TODO

## 3. Sets and parameters

Mapping between mathematical symbols and Python names.

| Symbol | Python name | Meaning |
|---|---|---|

In [ ]:
# TODO

## 4. Decision variables

$x_{rdg}$ (binary, only where $E_{rdg} = 1$); continuous shortfalls $h_{dg}$, $h^{S}_{dg}$, $h^{P}_{d}$, $h^{G}_{r}$; $v_{rd}$; $e^{\max}$, $e^{\min}$. Auxiliary expression $y_{rd} = \sum_g x_{rdg}$, with $y_{rd} = Y_{rd}$ for $d \in \{-1, 0\}$.

In [ ]:
# TODO

## 5. Constraints

### (1) Coverage

$$\sum_{r \in \mathcal{R}} x_{rdg} + h_{dg} = N_{dg} \qquad \forall d \in \mathcal{D}, \; g \in \mathcal{G}$$

Residents assigned plus uncovered positions equal the number required.

In [ ]:
# TODO

### (2a) Senior presence

$$\sum_{r \in \mathcal{R}_S} x_{rdg} + h^{S}_{dg} \ge S_{dg} \qquad \forall d, g$$

At least $S_{dg}$ seniors on each duty; $h^{S}_{dg}$ counts missing seniors.

In [ ]:
# TODO

### (2b) First-year limit

$$\sum_{r \in \mathcal{R}_1} x_{rdg} \le U_{dg} \qquad \forall d, g$$

At most $U_{dg}$ first-year residents on each duty.

In [ ]:
# TODO

### (3a) Eligibility and availability

$$x_{rdg} \le E_{rdg} \qquad \forall r, d, g$$

Implemented by creating $x_{rdg}$ only where $E_{rdg} = 1$.

In [ ]:
# TODO

### (3b) At most one duty per day

$$y_{rd} \le 1 \qquad \forall r, \; d \in \mathcal{D}$$

In [ ]:
# TODO

### (4) Rest after a duty

$$y_{rd} + y_{r,d+1} \le 1 \qquad \forall r, \; d \in \{0, \dots, n-1\}$$

No duty on consecutive days; for $d = 0$, $Y_{r0} + y_{r1} \le 1$ links with the previous month.

In [ ]:
# TODO

### (5) Alternate-day duties (soft)

$$y_{rd} + y_{r,d+2} \le 1 + v_{rd} \qquad \forall r, \; d \in \{-1, \dots, n-2\}$$

$v_{rd} = 1$ when the resident is on duty on $d$ and $d+2$; penalised in $f_4$.

In [ ]:
# TODO

### (6a) Monthly minimum

$$\sum_{d \in \mathcal{D}} y_{rd} + h^{G}_{r} \ge G^{\min}_r \qquad \forall r$$

In [ ]:
# TODO

### (6b) Monthly maximum

$$\sum_{d \in \mathcal{D}} y_{rd} \le G^{\max}_r \qquad \forall r$$

In [ ]:
# TODO

### (7) Requested days off

No constraint: $f_2 = \sum_{(r,d) \in \mathcal{P}} Q_{rd} \, y_{rd}$.

$y_{rd} = 1$ on a requested day off is the violation of the request.

In [ ]:
# TODO

### (8) Fairness of weekends and holidays

$$e^{\min} \le \frac{H_r + \sum_{d \in \mathcal{F}} y_{rd}}{\Phi_r} \le e^{\max} \qquad \forall r \in \mathcal{R}^F$$

In [ ]:
# TODO

### (9) Residents available for ordinary activity

Scenario A: $$\sum_{r} ROT_{rd} (1 - y_{r,d-1}) + h^{P}_{d} \ge PL_d \qquad \forall d \in \mathcal{L}$$

Scenario B: $$\sum_{r} ROT_{rd} (1 - y_{r,d-1} - y_{rd}) + h^{P}_{d} \ge PL_d \qquad \forall d \in \mathcal{L}$$

Post-call residents ($y_{r,d-1} = 1$) are not counted; in scenario B residents on duty on $d$ are not counted either.

In [ ]:
# TODO

### (10) Monthly duty hours (scenario B)

$$\sum_{d, g} T_g \, x_{rdg} \le 68 + 17 \, C_r \qquad \forall r$$

In [ ]:
# TODO

## 6. Objective (lexicographic)

$$\operatorname{lex\,min} \; (f_1, f_2, f_3, f_4)$$

- $f_1$: unmet requirements, $\sum_{d,g} (h_{dg} + h^{S}_{dg}) + \sum_{d \in \mathcal{L}} h^{P}_{d} + \sum_r h^{G}_{r}$
- $f_2$: violated requests, $\sum_{(r,d) \in \mathcal{P}} Q_{rd} \, y_{rd}$
- $f_3$: fairness range, $e^{\max} - e^{\min}$
- $f_4$: alternate-day patterns, $\sum_{r} \sum_{d=-1}^{n-2} v_{rd}$

Implemented with `model.setObjectiveN` (priorities 4, 3, 2, 1; zero degradation).

In [ ]:
# TODO

## 7. Solve and status handling

Check `model.Status` explicitly. An infeasible status means a data error (Section 6 of the formulation).

In [ ]:
# TODO

## 8. Solution validation

Check every constraint directly on the solution $x$, independently of the Gurobi model, and compute the KPIs from $x$ (not from the slack variables).

In [ ]:
# TODO

## 9. Initial results

Roster table, unmet requirements by type, violated requests, weekend/holiday loads, alternate-day patterns.

In [ ]:
# TODO